In [1]:
import requests
import pandas as pd

BASE_URL = "https://gamma-api.polymarket.com/markets"

def fetch_markets(limit=500):
    params = {
        "limit": limit,
        "offset": 10000
    }
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json()

data = fetch_markets()
# print(data[0].keys())
# keys = list(data[0].keys())
# for key in keys:
#     print(key, data[0].get(key))
print(len(data))

500


In [2]:
import json

def is_binary_market(market):
    outcomes = market.get("outcomes", [])
    outcomes = json.loads(outcomes)
    return len(outcomes) == 2

def is_resolved(market):
    outcomePricesStr = json.loads(market.get("outcomePrices"))
    outcomePrices = [float(x) for x in outcomePricesStr]
    return (outcomePrices == [0.0, 1.0] or outcomePrices == [1.0, 0.0])

def is_valid_market(market):
    return (
        is_binary_market(market) and
        is_resolved(market) and
        (market.get("volumeNum", 0) > 0)
    )

filtered = [m for m in data if is_valid_market(m)]

print("Filtered markets:", len(filtered))

Filtered markets: 492


In [3]:
from datetime import datetime

def get_outcome_label(market):
    outcomes = json.loads(market["outcomes"])
    prices = [float(x) for x in json.loads(market["outcomePrices"])]

    winner_index = prices.index(1.0)
    return 1 if outcomes[winner_index] == "Yes" else 0

def get_yes_token(market):
    outcomes = json.loads(market["outcomes"])
    token_ids = json.loads(market["clobTokenIds"])
    
    yes_index = outcomes.index("Yes")
    return token_ids[yes_index]

def parse_end_date(market):
    return datetime.fromisoformat(market["endDate"].replace("Z", "+00:00"))

In [34]:
yesno_data = []

for m in filtered:
    try:
        yesno_data.append({
            "id": m["id"],
            "question": m["question"],
            "end_time": parse_end_date(m),
            "yes_token": get_yes_token(m),
            "outcome": get_outcome_label(m),
            "volume": m["volumeNum"]
        })
    except Exception:
        continue

df = pd.DataFrame(yesno_data)

print(df.head())
print("Total yes/no markets:", len(df))

       id                                         question  \
0  255333  Will X remove likes + reposts counter in March?   
1  255335                    Fed rate cut by September 18?   
2  255336                      Fed rate cut by November 7?   
3  255337                     Fed rate cut by December 18?   
4  255338       Kung Fu Panda 4 over $48m opening weekend?   

                   end_time  \
0 2024-03-31 00:00:00+00:00   
1 2024-09-18 00:00:00+00:00   
2 2024-11-07 00:00:00+00:00   
3 2024-12-18 00:00:00+00:00   
4 2024-03-11 00:00:00+00:00   

                                           yes_token  outcome        volume  
0  3905029632436444580964166468467987293649111562...        0  1.140370e+04  
1  5037672276798297679281948332167547618468350597...        1  2.034532e+07  
2  9322802361841049010117128229226292399515496085...        1  2.020983e+06  
3  1143487640821006712994958593113021031697013551...        1  2.269863e+06  
4  3383646173658436364141420174674086475282997676.

In [51]:
HISTORY_URL = "https://clob.polymarket.com/trades"

def fetch_price_history(token_id):
    params = {
        "token_id": token_id,
        "limit": 1000
    }
    response = requests.get(HISTORY_URL, params=params)
    response.raise_for_status()
    return response.json()

In [52]:
for i in range(50):
    sample = df.iloc[i]
    history = fetch_price_history(sample["yes_token"])
    print(i, history)

HTTPError: 401 Client Error: Unauthorized for url: https://clob.polymarket.com/trades?token_id=39050296324364445809641664684679872936491115626778324924525327416420800978175&limit=1000